In [1]:
import pandas as pd
import numpy as np
import json
import ast
import os

In [ ]:
PATHSAVE = '../Data/PosProcessed/xlsx'
PATHLOAD = '../Data/PreProcessed/'
PATHJSON = '../Data/Raw/VarDict_v2.json'

---

#### **01 ) - Instânciando Dados**

---

---

##### . ) ...

In [3]:
df_vars_standardized = pd.read_csv(os.path.join(PATHSAVE, 'gemma3_12b_3bat_adt_v3.csv'))
df_vars_standardized.head()

,id,Col,Col_Standardized
0,CRED-002,status,INSTITUTIONAL and FINANCIAL
1,CRED-002,duration,INSTITUTIONAL and FINANCIAL
2,CRED-002,credit_history,INSTITUTIONAL and FINANCIAL
3,CRED-002,purpose,INSTITUTIONAL and FINANCIAL
4,CRED-002,amount,INSTITUTIONAL and FINANCIAL


---

##### A ) Instânciando Variáveis Sensíveis

In [21]:
with open(PATHJSON, 'r', encoding='UTF-8') as f:
    var_dict = json.load(f)

print(type(var_dict))

<class 'dict'>


In [22]:
vars = [
    {"name_normalize": chave, "vars": valores}
    for chave, valores in var_dict.items()
]

df_vars = pd.DataFrame(vars)
df_vars.head()

,name_normalize,vars
0,age,"[age, Age, person_age, CustomerAge, Customer_A..."
1,gender,"[Gender, person_gender, personal_status_sex, C..."
2,education,"[education, Education, person_education, educa..."
3,marital_status,"[married, MaritalStatus, Marital_Status, Clien..."
4,number_of_dependents,"[dependents, no_of_dependents, NrOfDependants,..."


In [23]:
df_vars_tidy = df_vars[['name_normalize', 'vars']].explode('vars').rename(columns={'vars': 'var'}).copy()
df_vars_tidy = df_vars_tidy.reset_index(drop=True)
df_vars_tidy.head()

,name_normalize,var
0,age,age
1,age,Age
2,age,person_age
3,age,CustomerAge
4,age,Customer_Age


In [24]:
df_vars_tidy['var'].value_counts().sum()

np.int64(320)

---

##### B ) Instânciando Tabela de Dados

In [5]:
df_fair = pd.read_excel(os.path.join(PATHLOAD, '06_Datasets.xlsx'))
df_fair.head()

,id,Dataset_Title,URL,is_encrypted,Columns,Columns_Count,Rows_Count,Rows_With_Missing,Missing_Values,Numeric_Cols,Categorical_Cols,Sensitive_Columns,Memory_Usage_MB,Sparsity_Ratio_%,Dimensionality_Ratio,Has_Sensitive_Data,Sensitive_Count
0,CRED-002,South German Credit,https://www.kaggle.com/datasets/sid321axn/sout...,0,"['status', 'duration', 'credit_history', 'purp...",21,1000,0,0,3,18,"['status', 'personal_status_sex', 'age']",0.40,0.00,0.021000,True,3
1,CRED-004,Australian Credit Approval,https://www.kaggle.com/datasets/bfueojjsjdjsl/...,1,"['CustomerID', 'A1', 'A2', 'A3', 'A4', 'A5', '...",16,690,0,0,16,0,[],0.08,0.00,0.023188,False,0
2,CRED-005,Japanese Credit Screening,https://www.kaggle.com/datasets/xiangshan1989/...,1,"['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8...",16,690,126,435,7,9,[],0.09,3.94,0.023188,False,0
3,CRED-007,Polish Companies Bankruptcy,https://www.kaggle.com/datasets/stealthtechnol...,1,"['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8...",66,43405,23438,41322,66,0,[],21.86,1.44,0.001521,False,0
4,CRED-008,Qualitative Bankruptcy,https://www.kaggle.com/datasets/jagadeesh23/qu...,1,"['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'Class']",7,250,0,0,0,7,[],0.00,0.00,0.028000,False,0


---

#### **02 ) - Correlação das Tabelas**

---

In [8]:
df_fair['Columns'] = df_fair['Columns'].apply(ast.literal_eval)
lista_columns = df_fair['Columns'].to_list()

In [11]:
df_split_fair_tidy = df_fair[['id', 'Dataset_Title','URL', 'Columns']].explode('Columns').rename(columns={'Columns' : 'Col'}).copy()
df_split_fair_tidy.head()

,id,Dataset_Title,URL,Col
0,CRED-002,South German Credit,https://www.kaggle.com/datasets/sid321axn/sout...,status
0,CRED-002,South German Credit,https://www.kaggle.com/datasets/sid321axn/sout...,duration
0,CRED-002,South German Credit,https://www.kaggle.com/datasets/sid321axn/sout...,credit_history
0,CRED-002,South German Credit,https://www.kaggle.com/datasets/sid321axn/sout...,purpose
0,CRED-002,South German Credit,https://www.kaggle.com/datasets/sid321axn/sout...,amount


In [28]:
list_fair_id = df_fair['id'].unique()
list_vars_norm = df_vars['name_normalize'].unique()
vars_norm = []
notin = 0
for id in list_fair_id:
    l = []
    list_cols = df_split_fair_tidy[df_split_fair_tidy['id'] == id]['Col'].to_numpy()
    
    for name in list_vars_norm:
        list_cols_raw = df_vars_tidy[df_vars_tidy['name_normalize'] == name]['var'].values
        
        for col_raw in list_cols:
            if col_raw in list_cols_raw:
                l.append(name)
                break
    vars_norm.append({'id' : id, 'Columns_Normalized' : l})

In [29]:
df_buffer = pd.DataFrame(vars_norm)
df_buffer.head()

,id,Columns_Normalized
0,CRED-002,"[age, gender, marital_status, number_of_depend..."
1,CRED-004,[]
2,CRED-005,[]
3,CRED-007,[]
4,CRED-008,[]


In [30]:
df_split_fair = df_fair[['id','Dataset_Title','URL','is_encrypted','Columns','Columns_Count','Rows_Count','Rows_With_Missing','Missing_Values','Numeric_Cols','Categorical_Cols','Memory_Usage_MB']]

In [31]:
valores_mapeados = df_split_fair['id'].map(df_buffer.set_index('id')['Columns_Normalized'])

if 'Columns_Normalized' in df_split_fair.columns:
    df_split_fair = df_split_fair.drop(columns=['Columns_Normalized'])

df_split_fair.insert(loc=4, column='Columns_Normalized', value=valores_mapeados)
df_split_fair.to_excel(os.path.join(PATHSAVE, '06_Data_v2.xlsx'), index=False, engine='openpyxl')
df_split_fair.head()

,id,Dataset_Title,URL,is_encrypted,Columns_Normalized,Columns,Columns_Count,Rows_Count,Rows_With_Missing,Missing_Values,Numeric_Cols,Categorical_Cols,Memory_Usage_MB
0,CRED-002,South German Credit,https://www.kaggle.com/datasets/sid321axn/sout...,0,"[age, gender, marital_status, number_of_depend...","[status, duration, credit_history, purpose, am...",21,1000,0,0,3,18,0.40
1,CRED-004,Australian Credit Approval,https://www.kaggle.com/datasets/bfueojjsjdjsl/...,1,[],"[CustomerID, A1, A2, A3, A4, A5, A6, A7, A8, A...",16,690,0,0,16,0,0.08
2,CRED-005,Japanese Credit Screening,https://www.kaggle.com/datasets/xiangshan1989/...,1,[],"[A1, A2, A3, A4, A5, A6, A7, A8, A9, A10, A11,...",16,690,126,435,7,9,0.09
3,CRED-007,Polish Companies Bankruptcy,https://www.kaggle.com/datasets/stealthtechnol...,1,[],"[A1, A2, A3, A4, A5, A6, A7, A8, A9, A10, A11,...",66,43405,23438,41322,66,0,21.86
4,CRED-008,Qualitative Bankruptcy,https://www.kaggle.com/datasets/jagadeesh23/qu...,1,[],"[V1, V2, V3, V4, V5, V6, Class]",7,250,0,0,0,7,0.00


---

#### **03 ) -**

---

In [ ]:
df_consolidado = df_fair[df_fair['is_encrypted'] == 0][['id', 'URL', 'Columns']].explode('Columns')
df_consolidado.rename(columns={'Columns': 'Col'}, inplace=True)

mapeamento = df_vars_standardized.set_index(['id', 'Col'])['Col_Standardized'].to_dict()

df_consolidado['Col_Standardized'] = df_consolidado.apply(
    lambda row: mapeamento.get((row['id'], row['Col']), row['Col']), axis=1
)

tabela_final = df_consolidado[['id', 'URL', 'Col', 'Col_Standardized']]

In [22]:
tabela_final.head()

,id,URL,Col,Col_Standardized
0,CRED-002,https://www.kaggle.com/datasets/sid321axn/sout...,status,INSTITUTIONAL and FINANCIAL
0,CRED-002,https://www.kaggle.com/datasets/sid321axn/sout...,duration,INSTITUTIONAL and FINANCIAL
0,CRED-002,https://www.kaggle.com/datasets/sid321axn/sout...,credit_history,INSTITUTIONAL and FINANCIAL
0,CRED-002,https://www.kaggle.com/datasets/sid321axn/sout...,purpose,INSTITUTIONAL and FINANCIAL
0,CRED-002,https://www.kaggle.com/datasets/sid321axn/sout...,amount,INSTITUTIONAL and FINANCIAL
